In [83]:
import pandas as pd

comp = pd.read_csv("data/comp trial 1.csv", low_memory=False)
fjc = pd.read_csv("data/fjc trial 1.csv", low_memory=False)

print(comp.shape, fjc.shape)
print(comp.head())
print(fjc.head())

(332314, 35) (841, 6)
  costat curcd datafmt indfmt consol  tic    datadate  gvkey      conm  \
0      A   USD     STD   INDL      C  AIR  31/05/2000   1004  AAR CORP   
1      A   USD     STD   INDL      C  AIR  31/05/2001   1004  AAR CORP   
2      A   USD     STD   INDL      C  AIR  31/05/2002   1004  AAR CORP   
3      A   USD     STD   INDL      C  AIR  31/05/2003   1004  AAR CORP   
4      A   USD     STD   INDL      C  AIR  31/05/2004   1004  AAR CORP   

      cik  ...     rect      seq    ebit      ni      sale    xint     xsga  \
0  1750.0  ...  128.348  339.515  70.658  35.163  1024.333  23.431  102.195   
1  1750.0  ...  115.187  340.212  45.790  18.531   874.255  21.887   96.077   
2  1750.0  ...   77.528  310.235   4.711 -58.939   638.721  19.798   85.037   
3  1750.0  ...   66.322  294.988   3.573 -12.410   606.337  19.539   78.845   
4  1750.0  ...  104.661  301.684  20.811   3.504   651.958  18.819   81.165   

     capx   oancf    mkvalt  
0  22.344  10.051  372.7519 

In [84]:
comp["datadate"] = pd.to_datetime(comp["datadate"], dayfirst=True, errors="coerce")
fjc["FILEDATE"] = pd.to_datetime(fjc["FILEDATE"], dayfirst=True, errors="coerce")

In [85]:
comp = comp[(comp["datadate"] >= "2000-01-01") & (comp["datadate"] <= "2018-12-31")]
fjc = fjc[(fjc["FILEDATE"] >= "2000-01-01") & (fjc["FILEDATE"] <= "2019-12-31")]

In [86]:
comp["cik"] = pd.to_numeric(comp["cik"], errors="coerce")
fjc["cik"] = pd.to_numeric(fjc["cik"], errors="coerce")

comp = comp.dropna(subset=["cik"]).copy()
fjc = fjc.dropna(subset=["cik"]).copy()

comp["cik"] = comp["cik"].astype(int)
fjc["cik"] = fjc["cik"].astype(int)

merged = comp.merge(fjc[["cik", "FILEDATE"]], on="cik", how="left")
merged["days"] = (merged["FILEDATE"] - merged["datadate"]).dt.days
merged["distress"] = ((merged["days"] > 0) & (merged["days"] <= 365)).astype(int)
merged.loc[merged["FILEDATE"].isna(), "distress"] = 0

merged_clean = merged.groupby(["gvkey", "datadate"], as_index=False).agg(distress=("distress", "max"))
final = comp.merge(merged_clean, on=["gvkey", "datadate"], how="left")
final["distress"] = final["distress"].fillna(0).astype(int)

print(len(comp), len(final))
print(final["distress"].value_counts())
final["year"] = final["datadate"].dt.year
print(final.groupby("year")["distress"].sum())


195302 195302
distress
0    195050
1       252
Name: count, dtype: int64
year
2000     1
2001     3
2002     1
2003     0
2004     8
2005     3
2006    13
2007    26
2008    26
2009    17
2010    24
2011    23
2012    14
2013    18
2014    16
2015    25
2016    16
2017    13
2018     5
Name: distress, dtype: int64


In [87]:

print(final["datadate"].min(), final["datadate"].max())
print(final["year"].min(), final["year"].max())
print(final.groupby("year").size())  # all firm-years per year, not just distress

2000-01-31 00:00:00 2018-12-31 00:00:00
2000 2018
year
2000    12154
2001    11692
2002    11434
2003    11230
2004    10969
2005    10814
2006    10535
2007    10205
2008     9894
2009     9769
2010     9744
2011     9785
2012    10060
2013    10116
2014     9846
2015     9524
2016     9381
2017     9148
2018     9002
dtype: int64
